# LeetCode 1657 — Determine if Two Strings Are Close bằng Python

Link đề: https://leetcode.com/problems/determine-if-two-strings-are-close/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sorting, Counting

Skeleton LeetCode:

```python
class Solution:
    def closeStrings(self, word1: str, word2: str) -> bool:
        ...
```

Notebook đi từ **BFS trên mọi xâu sinh được bằng hai phép biến đổi** (vét cạn theo định nghĩa) → nhận ra hai **bất biến** (tập ký tự, đa tập tần suất) → cài đặt bằng `Counter` + `sorted`, bằng **mảng 26 phần tử**, và bằng **đếm tần suất của tần suất** không cần sắp xếp.

## 1. Mục tiêu học

1. Hiểu hai phép biến đổi: **hoán đổi vị trí** (đổi thứ tự tùy ý) và **đổi vai hai ký tự** (hoán đổi tần suất giữa hai ký tự đang có mặt).
2. Tìm ra **bất biến** của mỗi phép: phép 1 giữ nguyên tần suất từng ký tự; phép 2 giữ nguyên **tập ký tự** và **đa tập tần suất**.
3. Chứng minh điều ngược lại: hai bất biến đó bằng nhau thì **luôn** biến đổi được — nên đó là điều kiện **cần và đủ**.
4. Cài đặt `O(n)` bằng `Counter`, mảng đếm 26 ô, hoặc `Counter` của các tần suất.
5. Kiểm thử bằng vét cạn BFS trên xâu ngắn và test ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Hai xâu gọi là **gần nhau** (close) nếu có thể biến xâu này thành xâu kia bằng các phép:
- **Phép 1:** hoán đổi **hai ký tự bất kỳ** (theo vị trí). Ví dụ `abcde → aecdb`.
- **Phép 2:** biến **mọi** lần xuất hiện của một ký tự **đang có** thành một ký tự **đang có** khác, đồng thời làm ngược lại với ký tự kia. Ví dụ `aacabb → bbcbaa` (mọi `a` thành `b`, mọi `b` thành `a`).

Được dùng các phép trên **bao nhiêu lần tùy ý**, trên **bất kỳ** xâu nào. Cho `word1`, `word2`, trả về `True` nếu chúng gần nhau, ngược lại `False`.

**Ví dụ 1:** `word1 = "abc"`, `word2 = "bca"` → `True` (hai lần phép 1: `abc → acb → bca`).

**Ví dụ 2:** `word1 = "a"`, `word2 = "aa"` → `False` (độ dài khác nhau, không phép nào đổi độ dài).

**Ví dụ 3:** `word1 = "cabbba"`, `word2 = "abbccc"` → `True`:
- Phép 1: `cabbba → caabbb`
- Phép 2 (đổi `b` ↔ `c`): `caabbb → baaccc`
- Phép 2 (đổi `a` ↔ `b`): `baaccc → abbccc`

**Ràng buộc:**
- `1 ≤ word1.length, word2.length ≤ 10⁵`
- Hai xâu chỉ gồm chữ cái tiếng Anh in thường.

In [1]:
import random
import time
from collections import Counter, deque
from itertools import permutations
from typing import List

examples = [("abc", "bca", True), ("a", "aa", False), ("cabbba", "abbccc", True)]

def op1(s, i, j):
    """Hoán đổi hai vị trí i, j."""
    t = list(s); t[i], t[j] = t[j], t[i]
    return ''.join(t)

def op2(s, x, y):
    """Đổi vai hai ký tự đang có mặt x, y."""
    assert x in s and y in s
    return ''.join(y if c == x else x if c == y else c for c in s)

s = "cabbba"
s1 = ''.join(sorted(s, key=lambda c: "cab".index(c)))      # sắp lại bằng nhiều lần phép 1
print("phép 1 (nhiều lần):", s, "->", s1)
s2 = op2(s1, 'b', 'c'); print("phép 2 (b<->c):", s1, "->", s2)
s3 = op2(s2, 'a', 'b'); print("phép 2 (a<->b):", s2, "->", s3)
print("về đúng word2?", s3 == "abbccc")

phép 1 (nhiều lần): cabbba -> caabbb
phép 2 (b<->c): caabbb -> baaccc
phép 2 (a<->b): baaccc -> abbccc
về đúng word2? True


## 3. Kiến thức nền

### 3.1 `Counter`: đếm tần suất

`Counter(word)` cho `{ký tự: số lần}`. `.keys()` là tập ký tự xuất hiện, `.values()` là các tần suất.

In [2]:
c1, c2 = Counter("cabbba"), Counter("abbccc")
print(c1, "|", c2)
print("tập ký tự:", set(c1), set(c2))
print("tần suất :", list(c1.values()), list(c2.values()))

Counter({'b': 3, 'a': 2, 'c': 1}) | Counter({'c': 3, 'b': 2, 'a': 1})
tập ký tự: {'a', 'c', 'b'} {'c', 'a', 'b'}
tần suất : [1, 2, 3] [1, 2, 3]


### 3.2 So sánh **đa tập** (multiset)

Hai list có cùng các phần tử (kể cả số lần lặp) ⇔ `sorted(a) == sorted(b)` ⇔ `Counter(a) == Counter(b)`. Lưu ý `set(a) == set(b)` **bỏ mất** số lần lặp — không đủ.

In [3]:
a, b = [3, 2, 1], [1, 3, 2]
print(sorted(a) == sorted(b), Counter(a) == Counter(b))
a, b = [1, 1, 2], [1, 2, 2]
print("set:", set(a) == set(b), "| sorted:", sorted(a) == sorted(b), "<- set không đủ để so đa tập")

True True
set: True | sorted: False <- set không đủ để so đa tập


### 3.3 Mảng đếm 26 ô và `ord`

`ord(c) - ord('a')` đổi chữ in thường thành chỉ số `0..25`. Mảng `cnt[26]` thay cho `Counter` — nhanh và có thứ tự cố định.

In [4]:
def count26(w):
    cnt = [0] * 26
    for ch in w:
        cnt[ord(ch) - 97] += 1
    return cnt

print(count26("cabbba")[:5], "| có mặt:", [chr(97 + i) for i, v in enumerate(count26("cabbba")) if v])

[2, 3, 1, 0, 0] | có mặt: ['a', 'b', 'c']


### 3.4 Bitmask cho tập ký tự

Tập ký tự có mặt có thể mã hóa thành một số nguyên 26 bit: bit `i` bật nếu chữ thứ `i` xuất hiện. So sánh hai tập = so sánh hai số.

In [5]:
def mask(w):
    m = 0
    for ch in w:
        m |= 1 << (ord(ch) - 97)
    return m

print(bin(mask("cabbba")), bin(mask("abbccc")), mask("cabbba") == mask("abbccc"), mask("abd") == mask("abc"))

0b111 0b111 True False


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 3:

| | a | b | c |
|---|---|---|---|
| `cabbba` | 2 | 3 | 1 |
| `abbccc` | 1 | 2 | 3 |

- **Phép 1** chỉ đổi **vị trí** ⇒ tần suất từng chữ không đổi; với phép 1 ta có thể sắp xếp xâu theo bất kỳ thứ tự nào ⇒ chỉ **bảng tần suất** là quan trọng.
- **Phép 2** đổi vai `x ↔ y` ⇒ **hoán đổi hai cột** tần suất trong bảng. Chữ nào có mặt vẫn có mặt (không tạo chữ mới, không xóa chữ nào).

Vậy với bảng `(a:2, b:3, c:1)` ta có thể hoán vị tùy ý các con số giữa các cột `a, b, c` ⇒ đạt được `(a:1, b:2, c:3)` ⇔ hai bảng có:
1. **Cùng tập cột** (cùng tập ký tự): `{a, b, c}` — ✔.
2. **Cùng đa tập các con số**: `{2, 3, 1}` và `{1, 2, 3}` — ✔.

Ví dụ phản: `"aab"` vs `"bbc"`: đa tập `{2, 1}` giống nhau, nhưng tập ký tự `{a, b}` ≠ `{b, c}` ⇒ `False` (không thể tạo ra chữ `c` từ `"aab"`).

In [6]:
def table(w):
    c = Counter(w)
    return {ch: c[ch] for ch in sorted(c)}

for w1, w2 in [("cabbba", "abbccc"), ("aab", "bbc"), ("aabb", "aaab"), ("abc", "bca")]:
    t1, t2 = table(w1), table(w2)
    same_keys = set(t1) == set(t2)
    same_multiset = sorted(t1.values()) == sorted(t2.values())
    print(f"{w1:>7} {t1} | {w2:>7} {t2} -> cùng tập chữ: {same_keys}, cùng đa tập tần suất: {same_multiset}")

 cabbba {'a': 2, 'b': 3, 'c': 1} |  abbccc {'a': 1, 'b': 2, 'c': 3} -> cùng tập chữ: True, cùng đa tập tần suất: True
    aab {'a': 2, 'b': 1} |     bbc {'b': 2, 'c': 1} -> cùng tập chữ: False, cùng đa tập tần suất: True
   aabb {'a': 2, 'b': 2} |    aaab {'a': 3, 'b': 1} -> cùng tập chữ: True, cùng đa tập tần suất: False
    abc {'a': 1, 'b': 1, 'c': 1} |     bca {'a': 1, 'b': 1, 'c': 1} -> cùng tập chữ: True, cùng đa tập tần suất: True


## 5. Cách 1 — Brute Force / baseline: BFS trên mọi xâu sinh được

**Ý tưởng:** xem mỗi xâu là một đỉnh, mỗi phép biến đổi là một cạnh. Từ `word1`, BFS sinh mọi xâu tới được; trả lời `word2` có trong tập đó không. Hai phép đều **khả nghịch** (làm lại lần nữa là về như cũ) nên quan hệ "gần nhau" là đối xứng — chỉ cần BFS từ một phía.

In [7]:
def close_bfs(word1, word2, limit=200000):
    if len(word1) != len(word2):
        return False
    seen = {word1}
    q = deque([word1])
    while q:
        s = q.popleft()
        if s == word2:
            return True
        n = len(s)
        nxt = [op1(s, i, j) for i in range(n) for j in range(i + 1, n)]
        chars = sorted(set(s))
        nxt += [op2(s, x, y) for i, x in enumerate(chars) for y in chars[i + 1:]]
        for t in nxt:
            if t not in seen:
                seen.add(t); q.append(t)
                if len(seen) > limit:
                    raise RuntimeError("quá nhiều trạng thái")
    return False

for w1, w2, exp in examples:
    print(w1, w2, "->", close_bfs(w1, w2), "(kỳ vọng", exp, ")")

abc bca -> True (kỳ vọng True )
a aa -> False (kỳ vọng False )
cabbba abbccc -> True (kỳ vọng True )


**Độ phức tạp:** số xâu tới được có thể tới `n!` (mọi hoán vị) nhân số cách đổi vai — hàm mũ. Mỗi đỉnh sinh `O(n² + k²)` láng giềng, mỗi láng giềng tốn `O(n)` để tạo. Chỉ dùng được cho `n ≤ 7`.

**Nút thắt:** ta đang phân biệt những xâu **chỉ khác thứ tự** — trong khi phép 1 cho phép đổi thứ tự tự do. Cần thu gọn mỗi xâu về một "dấu vân tay" bất biến.

## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến:**
- Phép 1 giữ nguyên `Counter(word)`.
- Phép 2 (đổi vai `x ↔ y`) biến `Counter` thành `Counter` với hai giá trị của `x`, `y` **hoán đổi** ⇒ giữ nguyên **tập khóa** và **đa tập giá trị**.

⇒ **Nếu** hai xâu gần nhau **thì** chúng có cùng tập ký tự và cùng đa tập tần suất (điều kiện **cần**).

**Đủ:** **nếu** cùng tập ký tự `K` và cùng đa tập tần suất **thì** tồn tại một hoán vị `π` trên `K` biến bảng tần suất của `word1` thành của `word2`. Mọi hoán vị là tích các **chuyển vị** (đổi chỗ hai phần tử) ⇒ thực hiện bằng một dãy phép 2. Sau đó hai xâu có cùng `Counter` ⇒ là hoán vị của nhau ⇒ dùng phép 1 để sắp lại thứ tự.

**Quy tắc kiểm tra:** `len(word1) == len(word2)` (hệ quả của đa tập tần suất bằng nhau) **và** `set(word1) == set(word2)` **và** `sorted(tần suất 1) == sorted(tần suất 2)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` + `sorted` + `set`

In [8]:
def close_counter(word1, word2):
    c1, c2 = Counter(word1), Counter(word2)
    return set(c1) == set(c2) and sorted(c1.values()) == sorted(c2.values())

class SolutionCounter:
    def closeStrings(self, word1: str, word2: str) -> bool:
        if len(word1) != len(word2):            # thoát sớm, không bắt buộc
            return False
        c1, c2 = Counter(word1), Counter(word2)
        return c1.keys() == c2.keys() and sorted(c1.values()) == sorted(c2.values())

for w1, w2, exp in examples:
    print(w1, w2, "->", close_counter(w1, w2), SolutionCounter().closeStrings(w1, w2), "(kỳ vọng", exp, ")")

abc bca -> True True (kỳ vọng True )
a aa -> False False (kỳ vọng False )
cabbba abbccc -> True True (kỳ vọng True )


`c1.keys() == c2.keys()` so sánh như **tập** (đối tượng `dict_keys` hỗ trợ phép so sánh kiểu tập). **Độ phức tạp:** `O(n)` để đếm + `O(26 log 26)` để sắp ⇒ `O(n)`; bộ nhớ `O(26)`.

### 7.2 Cách 3 — Hai mảng đếm 26 ô

In [9]:
def close_arrays(word1, word2):
    if len(word1) != len(word2):
        return False
    a, b = [0] * 26, [0] * 26
    for ch in word1:
        a[ord(ch) - 97] += 1
    for ch in word2:
        b[ord(ch) - 97] += 1
    for i in range(26):
        if (a[i] == 0) != (b[i] == 0):          # một bên có chữ i, bên kia không
            return False
    return sorted(a) == sorted(b)

for w1, w2, exp in examples:
    print(w1, w2, "->", close_arrays(w1, w2), "(kỳ vọng", exp, ")")

abc bca -> True (kỳ vọng True )
a aa -> False (kỳ vọng False )
cabbba abbccc -> True (kỳ vọng True )


Mẹo: sắp **cả mảng 26 ô** (gồm các số 0) vẫn đúng, vì số lượng số 0 bằng nhau khi tập ký tự bằng nhau. **Độ phức tạp:** `O(n + 26 log 26)`, bộ nhớ `O(26)`.

### 7.3 Cách 4 — `Counter` của các tần suất (không sắp xếp)

So sánh đa tập tần suất bằng `Counter(c1.values()) == Counter(c2.values())`, kết hợp so sánh tập ký tự bằng **bitmask**.

In [10]:
def close_freq_of_freq(word1, word2):
    c1, c2 = Counter(word1), Counter(word2)
    m1 = m2 = 0
    for ch in c1: m1 |= 1 << (ord(ch) - 97)
    for ch in c2: m2 |= 1 << (ord(ch) - 97)
    return m1 == m2 and Counter(c1.values()) == Counter(c2.values())

for w1, w2, exp in examples:
    print(w1, w2, "->", close_freq_of_freq(w1, w2), "(kỳ vọng", exp, ")")

abc bca -> True (kỳ vọng True )
a aa -> False (kỳ vọng False )
cabbba abbccc -> True (kỳ vọng True )


**Độ phức tạp:** `O(n)` thời gian, `O(26)` bộ nhớ — không cần sắp xếp (có ích khi bảng chữ cái **lớn**, ví dụ Unicode).

### 7.4 Cách 5 — Dựng tường minh dãy phép biến đổi (để "thấy" tính đủ)

Không cần cho LeetCode, nhưng giúp hiểu chứng minh: tìm một dãy phép 2 biến bảng tần suất của `word1` thành của `word2` (như sắp xếp chọn — mỗi bước đặt đúng tần suất cho một chữ), rồi dùng phép 1 sắp lại vị trí.

In [11]:
def construct_ops(word1, word2):
    """Trả về danh sách (mô tả phép, xâu sau phép) biến word1 thành word2, hoặc None nếu không gần nhau."""
    if not close_counter(word1, word2):
        return None
    ops = []
    s = word1
    target = Counter(word2)
    for ch in sorted(target):
        cur = Counter(s)
        if cur[ch] != target[ch]:
            # tìm chữ khác đang có tần suất mong muốn của ch và chưa "chốt"
            other = next(o for o in sorted(cur) if o > ch and cur[o] == target[ch])
            s = op2(s, ch, other)
            ops.append((f"phép 2: {ch} <-> {other}", s))
    # sắp lại vị trí bằng phép 1 (selection-sort theo word2)
    t = list(s)
    for i in range(len(t)):
        if t[i] != word2[i]:
            j = next(j for j in range(i + 1, len(t)) if t[j] == word2[i])
            t[i], t[j] = t[j], t[i]
            ops.append((f"phép 1: đổi vị trí {i}, {j}", ''.join(t)))
    return ops

for step, st in construct_ops("cabbba", "abbccc"):
    print(f"{step:<24} -> {st}")

phép 2: a <-> c          -> acbbbc
phép 2: b <-> c          -> abcccb
phép 1: đổi vị trí 2, 5  -> abbccc


Thứ tự xử lý chữ tăng dần + chỉ chọn `other > ch` bảo đảm những chữ đã "chốt" không bị đụng lại; vì đa tập tần suất bằng nhau nên luôn tìm được `other`. Số phép 2 ≤ 25, số phép 1 ≤ `n − 1`.

## 8. TRACE từng bước

Trace cách 3 (mảng đếm) trên ví dụ 3 và trên phản ví dụ `("aab", "bbc")`:

In [12]:
def trace_arrays(word1, word2):
    print(f"word1={word1!r}, word2={word2!r}")
    if len(word1) != len(word2):
        print("  độ dài khác nhau -> False"); return False
    a, b = [0] * 26, [0] * 26
    for ch in word1: a[ord(ch) - 97] += 1
    for ch in word2: b[ord(ch) - 97] += 1
    used = [i for i in range(26) if a[i] or b[i]]
    print("  chữ   :", [chr(97 + i) for i in used])
    print("  word1 :", [a[i] for i in used])
    print("  word2 :", [b[i] for i in used])
    for i in used:
        if (a[i] == 0) != (b[i] == 0):
            print(f"  chữ {chr(97 + i)!r} chỉ có ở một xâu -> False"); return False
    sa, sb = sorted(x for x in a if x), sorted(x for x in b if x)
    print("  đa tập tần suất:", sa, "vs", sb, "->", sa == sb)
    return sa == sb

trace_arrays("cabbba", "abbccc")
trace_arrays("aab", "bbc")
trace_arrays("aabb", "aaab")

word1='cabbba', word2='abbccc'
  chữ   : ['a', 'b', 'c']
  word1 : [2, 3, 1]
  word2 : [1, 2, 3]
  đa tập tần suất: [1, 2, 3] vs [1, 2, 3] -> True
word1='aab', word2='bbc'
  chữ   : ['a', 'b', 'c']
  word1 : [2, 1, 0]
  word2 : [0, 2, 1]
  chữ 'a' chỉ có ở một xâu -> False
word1='aabb', word2='aaab'
  chữ   : ['a', 'b']
  word1 : [2, 2]
  word2 : [3, 1]
  đa tập tần suất: [2, 2] vs [1, 3] -> False


False

Bảng trạng thái cho ví dụ 3:

| bước | kiểm tra | kết quả |
|---|---|---|
| 1 | độ dài `6 == 6` | ✔ |
| 2 | tập chữ `{a,b,c}` vs `{a,b,c}` | ✔ |
| 3 | đa tập `[1,2,3]` vs `[1,2,3]` | ✔ ⇒ `True` |

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def closeStrings(self, word1: str, word2: str) -> bool:
        if len(word1) != len(word2):
            return False
        c1, c2 = Counter(word1), Counter(word2)
        return set(c1) == set(c2) and sorted(c1.values()) == sorted(c2.values())

for w1, w2, exp in examples:
    assert Solution().closeStrings(w1, w2) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `K(w)` là tập ký tự của `w`, `F(w)` là đa tập tần suất.

**Cần.** Phép 1 không đổi `Counter(w)` ⇒ không đổi `K`, `F`. Phép 2 với `x, y ∈ K(w)`: ký tự `x` nhận tần suất cũ của `y` và ngược lại, các chữ khác giữ nguyên ⇒ `K` không đổi (cả hai vẫn có mặt), `F` không đổi (chỉ hoán đổi hai giá trị). Vậy nếu `word1` biến được thành `word2` thì `K(word1) = K(word2)` và `F(word1) = F(word2)`.

**Đủ.** Giả sử `K(word1) = K(word2) = K` và `F(word1) = F(word2)`. Khi đó có một song ánh `π: K → K` với `cnt1[c] = cnt2[π(c)]`. Viết `π` thành tích các chuyển vị; mỗi chuyển vị `(x y)` là một phép 2 hợp lệ (vì `x, y ∈ K` đều đang có mặt). Sau dãy phép 2, xâu thu được có `Counter` bằng `Counter(word2)` ⇒ là một hoán vị của `word2` ⇒ một dãy phép 1 (đổi chỗ) sắp lại đúng thứ tự. (Cách 5 dựng tường minh dãy này.)

**Độ dài:** `F` bằng nhau ⇒ tổng tần suất bằng nhau ⇒ cùng độ dài, nên kiểm tra `len` ở đầu chỉ là thoát sớm.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(word1) + len(word2)`, bảng chữ cái `Σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS trên xâu | hàm mũ (tới `n!`) | hàm mũ |
| 2. Counter + sorted + set | `O(n + Σ log Σ)` | `O(Σ)` |
| 3. Hai mảng 26 ô | `O(n + Σ log Σ)` | `O(Σ)` |
| 4. Counter của tần suất + bitmask | `O(n + Σ)` | `O(Σ)` |
| 5. Dựng dãy phép (minh họa) | `O(Σ · n + n²)` | `O(n)` |

Đo trên `n = 10⁵` mỗi xâu:

In [14]:
rnd = random.Random(1657)
w1 = ''.join(rnd.choice("abcdefghij") for _ in range(10**5))
perm = dict(zip("abcdefghij", "jihgfedcba"))
w2 = ''.join(perm[c] for c in w1)
w2 = ''.join(rnd.sample(w2, len(w2)))
for name, fn in [("counter", close_counter), ("arrays", close_arrays), ("freq_of_freq", close_freq_of_freq), ("Solution", Solution().closeStrings)]:
    t0 = time.perf_counter(); r = fn(w1, w2); dt = time.perf_counter() - t0
    print(f"{name:<13}: {r}  {dt * 1000:6.1f} ms")

counter      : True     4.6 ms
arrays       : True     6.6 ms
freq_of_freq : True     4.7 ms
Solution     : True     4.7 ms


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: một ký tự, độ dài khác, cùng tần suất khác tập chữ, cùng tập chữ khác tần suất, giống hệt.
3. Đối chiếu với BFS vét cạn trên **mọi cặp** xâu ngắn trên bảng chữ `{a, b, c}`.
4. Ngẫu nhiên cỡ lớn: sinh cặp **chắc chắn gần nhau** (đổi vai ngẫu nhiên + xáo trộn) và cặp "suýt gần" (đổi một ký tự).

In [15]:
ALL = {
    "counter": close_counter,
    "arrays": close_arrays,
    "freq_of_freq": close_freq_of_freq,
    "SolCounter": lambda a, b: SolutionCounter().closeStrings(a, b),
    "Solution": lambda a, b: Solution().closeStrings(a, b),
}

def check(w1, w2, exp=None, brute=True):
    ref = close_bfs(w1, w2) if brute else close_counter(w1, w2)
    for name, fn in ALL.items():
        assert fn(w1, w2) == ref, (name, w1, w2)
        assert fn(w2, w1) == ref, (name, "đối xứng", w1, w2)
    if exp is not None:
        assert ref == exp, (w1, w2, ref, exp)
    if ref:
        ops = construct_ops(w1, w2)
        assert ops is not None and (ops[-1][1] if ops else w1) == w2
    return ref

for w1, w2, exp in examples:
    check(w1, w2, exp)
print("Ví dụ đề: OK")

for w1, w2, exp in [("a", "a", True), ("a", "b", False), ("ab", "ba", True), ("aab", "bbc", False),
                    ("aabb", "aaab", False), ("abbzzca", "babzzcz", False), ("uau", "ssx", False), ("aaabbc", "abbccc", True)]:
    check(w1, w2, exp)
print("Edge case: OK")

from itertools import product
words = [''.join(t) for n in range(1, 6) for t in product("abc", repeat=n)]
cnt = 0
by_len = {}
for w in words:
    by_len.setdefault(len(w), []).append(w)
rnd = random.Random(7)
for n, ws in by_len.items():
    sample = ws if len(ws) <= 30 else rnd.sample(ws, 30)
    for w1 in sample:
        for w2 in sample:
            check(w1, w2); cnt += 1
print(f"{cnt} cặp xâu ngắn trên {{a,b,c}} so với BFS vét cạn: OK")

Ví dụ đề: OK
Edge case: OK


2619 cặp xâu ngắn trên {a,b,c} so với BFS vét cạn: OK


In [16]:
rnd = random.Random(2018)
letters = "abcdefghijklmnopqrstuvwxyz"
for _ in range(300):
    k = rnd.randint(1, 26)
    alpha = rnd.sample(letters, k)
    w1 = ''.join(rnd.choice(alpha) for _ in range(rnd.randint(1, 2000)))
    present = sorted(set(w1))
    shuffled = present[:]; rnd.shuffle(shuffled)
    mp = dict(zip(present, shuffled))                 # đổi vai: hoán vị trên tập chữ có mặt
    w2 = list(mp[c] for c in w1); rnd.shuffle(w2); w2 = ''.join(w2)
    check(w1, w2, True, brute=False)
    # "suýt gần": đổi một ký tự sang chữ khác
    i = rnd.randrange(len(w2))
    w3 = w2[:i] + rnd.choice(letters) + w2[i + 1:]
    check(w1, w3, brute=False)
print("300 cặp chắc chắn gần nhau + 300 cặp bị nhiễu: OK")

300 cặp chắc chắn gần nhau + 300 cặp bị nhiễu: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ so đa tập tần suất, quên so tập ký tự

In [17]:
def close_bug_freq_only(w1, w2):
    return sorted(Counter(w1).values()) == sorted(Counter(w2).values())     # SAI: thiếu tập chữ

for w1, w2 in [("aab", "bbc"), ("uau", "ssx")]:
    print(f"{w1}, {w2}: sai = {close_bug_freq_only(w1, w2)}, đúng = {Solution().closeStrings(w1, w2)}")

aab, bbc: sai = True, đúng = False
uau, ssx: sai = True, đúng = False


**Vì sao sai:** phép 2 chỉ đổi giữa các chữ **đang có**, không tạo ra chữ mới. `"aab"` không bao giờ chứa `c`. **Sửa:** thêm `set(c1) == set(c2)`.

### Lỗi 2 — Chỉ so tập ký tự (và độ dài)

In [18]:
def close_bug_set_only(w1, w2):
    return len(w1) == len(w2) and set(w1) == set(w2)           # SAI: thiếu đa tập tần suất

for w1, w2 in [("aabb", "aaab"), ("aabbcc", "aaabbc")]:
    print(f"{w1}, {w2}: sai = {close_bug_set_only(w1, w2)}, đúng = {Solution().closeStrings(w1, w2)}")

aabb, aaab: sai = True, đúng = False
aabbcc, aaabbc: sai = True, đúng = False


**Vì sao sai:** tần suất `{2, 2}` không thể hoán vị thành `{3, 1}`. **Sửa:** thêm `sorted(c1.values()) == sorted(c2.values())`.

### Lỗi 3 — So `Counter` bằng nhau (chỉ đúng cho phép 1 — kiểm tra anagram)

In [19]:
def close_bug_anagram(w1, w2):
    return Counter(w1) == Counter(w2)                         # SAI: bỏ qua phép 2

w1, w2 = "cabbba", "abbccc"
print("anagram:", close_bug_anagram(w1, w2), "| đúng:", Solution().closeStrings(w1, w2))

anagram: False | đúng: True


**Vì sao sai:** đó là điều kiện cho **chỉ** phép 1. Phép 2 cho phép hoán đổi tần suất giữa các chữ. **Sửa:** so tập chữ + đa tập tần suất.

### Lỗi 4 — Dùng `set` của các tần suất thay cho đa tập

In [20]:
def close_bug_set_of_freqs(w1, w2):
    c1, c2 = Counter(w1), Counter(w2)
    return set(c1) == set(c2) and set(c1.values()) == set(c2.values())    # SAI: set bỏ lặp

w1, w2 = "abcc", "abbcc"
print("set tần suất:", close_bug_set_of_freqs(w1, w2), "| đúng:", Solution().closeStrings(w1, w2))
w1, w2 = "abccdddeee", "abbccddeee"          # cùng độ dài 10, tần suất [1,1,2,3,3] vs [1,2,2,2,3]
print("set tần suất:", close_bug_set_of_freqs(w1, w2), "| đúng:", Solution().closeStrings(w1, w2))

set tần suất: True | đúng: False
set tần suất: True | đúng: False


**Vì sao sai:** `{1, 1, 2}` và `{1, 2, 2}` có cùng `set` `{1, 2}` nhưng khác đa tập (và khác cả độ dài); ví dụ thứ hai còn **cùng độ dài** (`[1,1,2,3,3]` vs `[1,2,2,2,3]`) mà vẫn sai. **Sửa:** `sorted(...)` hoặc `Counter(...)` của các tần suất.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Có thể biến đổi bằng các phép … hay không?" ⇒ tìm **bất biến** của từng phép, rồi chứng minh bất biến là **đủ**.
- Phép "đổi chỗ tùy ý" ⇒ chỉ còn **tần suất** quan trọng (thứ tự vô nghĩa).
- Phép "đổi vai hai nhãn" ⇒ chỉ còn **đa tập** tần suất và **tập nhãn**.
- So sánh đa tập: `sorted(a) == sorted(b)` hoặc `Counter(a) == Counter(b)`; **đừng** dùng `set`.
- Bảng chữ nhỏ (26) ⇒ mảng đếm + bitmask là nhanh nhất.

## 15. Biến thể của bài

1. **Chỉ có phép 1:** kiểm tra **anagram** (LeetCode 242).
2. **Chỉ có phép 2:** hai xâu phải có **cùng cấu trúc vị trí** sau một song ánh nhãn — kiểm tra **isomorphic** (LeetCode 205) với điều kiện thêm "song ánh trên cùng tập chữ".
3. **Phép 2 cho phép đổi sang chữ chưa xuất hiện:** chỉ còn cần cùng đa tập tần suất.
4. **Đếm số phép tối thiểu:** số phép 2 tối thiểu = `|K| − số chu trình` của hoán vị cần thiết (phải chọn cách gán tối ưu khi có tần suất trùng); số phép 1 tối thiểu = `n − số chu trình` của hoán vị vị trí.
5. **Nhóm các xâu "gần nhau"** trong một danh sách: khóa băm `(frozenset(K), tuple(sorted(F)))`.

In [21]:
def group_close(words):
    groups = {}
    for w in words:
        c = Counter(w)
        key = (frozenset(c), tuple(sorted(c.values())))
        groups.setdefault(key, []).append(w)
    return list(groups.values())

def close_relabel_any(w1, w2):
    """Biến thể 3: phép 2 được đổi sang chữ bất kỳ (kể cả chưa có)."""
    return sorted(Counter(w1).values()) == sorted(Counter(w2).values())

print(group_close(["abc", "bca", "aab", "bba", "bbc", "cabbba", "abbccc", "xyz"]))
print("biến thể 3 ('aab', 'bbc'):", close_relabel_any("aab", "bbc"))

[['abc', 'bca'], ['aab', 'bba'], ['bbc'], ['cabbba', 'abbccc'], ['xyz']]
biến thể 3 ('aab', 'bbc'): True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | duyệt mọi xâu sinh được | hàm mũ | hàm mũ | kiểm chứng, xâu rất ngắn |
| 2. Counter + sorted | tập khóa + đa tập giá trị | `O(n)` | `O(Σ)` | **nộp bài**, ngắn gọn |
| 3. Mảng 26 ô | đếm + có/không + sorted | `O(n)` | `O(26)` | nhanh, không cần thư viện |
| 4. Counter của tần suất | bitmask + đếm tần suất | `O(n)` | `O(Σ)` | bảng chữ lớn, tránh sắp xếp |
| 5. Dựng dãy phép | minh họa chứng minh | `O(n²)` | `O(n)` | khi cần chỉ ra cách biến đổi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().closeStrings("abbzzca", "babzzcz")` và `Solution().closeStrings("aaabbc", "abbccc")`?

**Bài 2 (tính tay):** Lập bảng tần suất cho `word1 = "xxyyzz"`, `word2 = "zzxxyy"` và `word1 = "xxxyyz"`, `word2 = "xyyzzz"`; kết luận từng cặp.

**Bài 3 (điền TODO):** Hoàn thiện `close_key(word)` trả về một **khóa băm** sao cho hai xâu gần nhau ⇔ khóa bằng nhau.

**Bài 4 (tự viết lại):** Viết lại cách 3 chỉ dùng **một** mảng 26 ô cho tần suất mỗi xâu và **counting sort** (mảng đếm theo giá trị tần suất) thay cho `sorted`.

**Bài 5 (mini challenge):** Viết `min_relabel_ops(word1, word2)` trả về **số phép 2 tối thiểu** để biến bảng tần suất của `word1` thành của `word2` (giả sử hai xâu gần nhau). Kiểm bằng BFS trên các bảng tần suất cho bảng chữ nhỏ.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def close_key(word):
    # TODO: (frozenset của các chữ, tuple các tần suất đã sắp)
    pass

# Bài 4 — TODO
def close_counting_sort(word1, word2):
    # TODO: freq_of_freq[v] = số chữ có tần suất v (v <= n), so hai mảng này + so tập chữ
    pass

# Bài 5 — TODO
def min_relabel_ops(word1, word2):
    # TODO: với mỗi giá trị tần suất, các chữ mang giá trị đó trong word1 và word2;
    #       chữ nằm ở cả hai phía có thể "đứng yên"; đáp án = |K| - số chu trình tối đa
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: So tập chữ trước.
- Mức 2: `"abbzzca"` có `c`, `"babzzcz"` cũng có — so tiếp tần suất.
- Mức 3: Tần suất `{a:2, b:2, z:2, c:1}` vs `{a:1, b:2, z:3, c:1}` ⇒ đa tập `[1,2,2,2]` khác `[1,1,2,3]`.

**Bài 2**
- Mức 1: Cặp 1 là anagram.
- Mức 2: Cặp 2: `{x:3, y:2, z:1}` vs `{x:1, y:2, z:3}`.
- Mức 3: Cả hai cặp đều `True`.

**Bài 3**
- Mức 1: Khóa phải bất biến dưới hai phép.
- Mức 2: `frozenset` vì `set` không băm được.
- Mức 3: `tuple(sorted(...))` vì `list` không băm được.

**Bài 4**
- Mức 1: Tần suất ≤ `n`, nên đếm theo giá trị bằng `dict` hoặc mảng cỡ `n + 1`.
- Mức 2: Có thể chỉ cần `Counter` của tối đa 26 giá trị.
- Mức 3: Đừng quên kiểm tra tập chữ.

**Bài 5**
- Mức 1: Một hoán vị trên `m` phần tử với `c` chu trình cần `m − c` chuyển vị.
- Mức 2: Chữ `x` cùng tần suất ở cả hai xâu nên là điểm bất động (chu trình độ dài 1).
- Mức 3: Trong mỗi "lớp tần suất" `v`, gọi `A_v` = các chữ có tần suất `v` trong `word1`, `B_v` = trong `word2`. Gán cố định `A_v ∩ B_v`; phần còn lại ghép tối ưu — kiểm bằng BFS.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = (Solution().closeStrings("abbzzca", "babzzcz"), Solution().closeStrings("aaabbc", "abbccc"))
print("Bài 1:", r1)
assert r1 == (close_bfs("abbzzca", "babzzcz"), True) and r1[0] is False

# Bài 2
for a, b in [("xxyyzz", "zzxxyy"), ("xxxyyz", "xyyzzz")]:
    print(f"  {a} {table(a)} | {b} {table(b)} ->", Solution().closeStrings(a, b))
    assert Solution().closeStrings(a, b) and close_bfs(a, b)

# Bài 3
def close_key_sol(word):
    c = Counter(word)
    return frozenset(c), tuple(sorted(c.values()))

# Bài 4
def close_counting_sort_sol(word1, word2):
    if len(word1) != len(word2):
        return False
    a, b = [0] * 26, [0] * 26
    for ch in word1: a[ord(ch) - 97] += 1
    for ch in word2: b[ord(ch) - 97] += 1
    if any((x == 0) != (y == 0) for x, y in zip(a, b)):
        return False
    fa, fb = {}, {}
    for x in a:
        if x: fa[x] = fa.get(x, 0) + 1
    for y in b:
        if y: fb[y] = fb.get(y, 0) + 1
    return fa == fb

rnd = random.Random(3)
for _ in range(3000):
    w1 = ''.join(rnd.choice("abcd") for _ in range(rnd.randint(1, 8)))
    w2 = ''.join(rnd.choice("abcd") for _ in range(rnd.randint(1, 8)))
    exp = Solution().closeStrings(w1, w2)
    assert (close_key_sol(w1) == close_key_sol(w2)) == exp
    assert close_counting_sort_sol(w1, w2) == exp
print("Bài 3, Bài 4: OK")

Bài 1: (False, True)
  xxyyzz {'x': 2, 'y': 2, 'z': 2} | zzxxyy {'x': 2, 'y': 2, 'z': 2} -> True
  xxxyyz {'x': 3, 'y': 2, 'z': 1} | xyyzzz {'x': 1, 'y': 2, 'z': 3} -> True
Bài 3, Bài 4: OK


In [24]:
# Bài 5
from itertools import permutations as perms

def min_relabel_ops_sol(word1, word2):
    c1, c2 = Counter(word1), Counter(word2)
    K = sorted(c1)
    # thử mọi song ánh hợp lệ pi: K -> K với c1[x] == c2[pi(x)], lấy max số chu trình (bảng chữ nhỏ)
    best_cycles = -1
    for img in perms(K):
        pi = dict(zip(K, img))
        if any(c1[x] != c2[pi[x]] for x in K):
            continue
        seen, cycles = set(), 0
        for x in K:
            if x not in seen:
                cycles += 1
                while x not in seen:
                    seen.add(x); x = pi[x]
        best_cycles = max(best_cycles, cycles)
    return len(K) - best_cycles

def min_relabel_bfs(word1, word2):
    """BFS trên bảng tần suất (tuple theo thứ tự chữ), mỗi bước hoán đổi hai giá trị."""
    c1, c2 = Counter(word1), Counter(word2)
    K = sorted(c1)
    start, goal = tuple(c1[x] for x in K), tuple(c2[x] for x in K)
    dist = {start: 0}; q = deque([start])
    while q:
        t = q.popleft()
        if t == goal:
            return dist[t]
        for i in range(len(t)):
            for j in range(i + 1, len(t)):
                u = list(t); u[i], u[j] = u[j], u[i]; u = tuple(u)
                if u not in dist:
                    dist[u] = dist[t] + 1; q.append(u)

rnd = random.Random(5)
tested = 0
while tested < 400:
    w1 = ''.join(rnd.choice("abcde") for _ in range(rnd.randint(1, 12)))
    present = sorted(set(w1)); sh = present[:]; rnd.shuffle(sh)
    w2 = ''.join(dict(zip(present, sh))[c] for c in w1)
    assert min_relabel_ops_sol(w1, w2) == min_relabel_bfs(w1, w2)
    tested += 1
print("Bài 5 OK | cabbba -> abbccc cần", min_relabel_ops_sol("cabbba", "abbccc"), "phép 2")

Bài 5 OK | cabbba -> abbccc cần 2 phép 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "biến đổi được không" bằng bất biến:**

```python
c1, c2 = Counter(word1), Counter(word2)
return (len(word1) == len(word2)                  # thoát sớm
        and set(c1) == set(c2)                    # bất biến 1: tập nhãn
        and sorted(c1.values()) == sorted(c2.values()))   # bất biến 2: đa tập tần suất
```

**Khóa băm nhóm các xâu gần nhau:** `(frozenset(c), tuple(sorted(c.values())))`.

**Checklist:**
- [ ] So **tập ký tự** (phép 2 không tạo chữ mới)?
- [ ] So **đa tập** tần suất bằng `sorted`/`Counter`, không phải `set`?
- [ ] Không nhầm với kiểm tra anagram (`Counter(w1) == Counter(w2)`)?
- [ ] Hai xâu khác độ dài trả `False`?
- [ ] Đã đối chiếu với BFS vét cạn trên xâu ngắn?
- [ ] Chữ ký đúng: `def closeStrings(self, word1: str, word2: str) -> bool`?